# Train the U-Net on Colab

Before running:
1. Make `olivcho/ECE488` public on GitHub.
2. Choose **Runtime > Change runtime type > GPU**.

Run folders and checkpoints are saved to Google Drive, so they survive a disconnect.

## 1. Check the GPU and mount Drive

In [ ]:
!nvidia-smi -L

from google.colab import drive
drive.mount("/content/drive")

EXPERIMENTS_ROOT = "/content/drive/MyDrive/ece488/experiments"
!mkdir -p {EXPERIMENTS_ROOT}

## 2. Get the code


In [ ]:
import os
from pathlib import Path

BRANCH = "main"
REPO_DIR = Path("/content/ECE488")

if not REPO_DIR.exists():
    !git clone --quiet --branch {BRANCH} https://github.com/olivcho/ECE488.git {REPO_DIR}

os.chdir(REPO_DIR)
!git pull --quiet --ff-only
!git log -1 --format='%h %s'
!git status --short

## 3. Install dependencies

This keeps Colab's own PyTorch so it matches the GPU drivers. Each run records the versions it used.

In [ ]:
%pip install -q lpips torchmetrics datasets huggingface_hub pyyaml pytest
%pip install -q --no-deps -e .

# A first local download stalled on Hugging Face's Xet path, so use plain HTTPS.
os.environ["HF_HUB_DISABLE_XET"] = "1"

!python -m pytest -q

## 4. Short test

This trains the same small model on 500 images for 2 epochs. Check that previews look sensible, then use the printed seconds to estimate the training run:

`training epoch seconds ≈ test epoch seconds × 5,000 / 500`

In [ ]:
!python scripts/train_unet.py --config configs/unet_colab_test.yaml --experiments-root {EXPERIMENTS_ROOT}

## 5. Training run

This downloads all 7.6 GB, because the 1,000 validation images are drawn from every shard. It then trains a 16-channel, depth 3 U-Net on 5,000 images for 10 epochs. The test split is never used.

In [ ]:
!python scripts/train_unet.py --config configs/unet_small.yaml --experiments-root {EXPERIMENTS_ROOT}

## 6. Resume after a disconnect

Run sections 1 to 3 again, then set `RUN` to the folder printed by the training run. Training continues after the last finished epoch.

In [ ]:
RUN = f"{EXPERIMENTS_ROOT}/unet_small/YYMMDD_unet_v1"  # Replace with your run folder
!python scripts/train_unet.py --resume {RUN}

## 7. See the results

In [ ]:
import pandas as pd
from IPython.display import Image, display

RUN = f"{EXPERIMENTS_ROOT}/unet_small/YYMMDD_unet_v1"  # Replace with your run folder
display(pd.read_csv(f"{RUN}/metrics.csv"))
latest_preview = sorted(Path(RUN, "previews").glob("*.png"))[-1]
display(Image(str(latest_preview)))